# Factor Zoo LQ45 — explorer deskriptif

Notebook ini menghitung Value, Quality, Momentum, dan Low Volatility dari snapshot **Sectors.app yang sudah tersimpan lokal**. Skor lintas emiten berada pada rentang −3 sampai +3. Ini bukan model prediksi, rekomendasi transaksi, maupun pengubah bobot Portfolio Lab.

Semua sel sengaja belum dijalankan dan output disimpan kosong. Jalankan berurutan di Colab, Kaggle, atau Jupyter setelah melampirkan bundle proyek. Angka dan grafik baru muncul dari snapshot yang dilampirkan.

## Sumber dan provenance

Input adalah file company report, harga harian emiten, IHSG, universe LQ45, modul kalkulasi bersama, dan `manifest.json` berisi SHA-256. Bundle diekspor dari data lokal yang sudah disetujui; notebook ini **tidak** memanggil API Sectors, internet, atau OpenRouter. Keanggotaan LQ45 adalah snapshot sekarang, bukan sejarah keanggotaan point-in-time. Harga belum diverifikasi sudah disesuaikan untuk corporate action/dividen.

## Lampirkan bundle lokal — Colab / Kaggle / Jupyter

Buat `portfolio_lab_bundle.zip` dengan `scripts/export_portfolio_lab_notebook_data.py` di checkout proyek, lalu unggah file itu ke Colab atau lampirkan sebagai dataset Kaggle. Di Jupyter, letakkan bundle di direktori kerja atau ubah `BUNDLE_PATH` pada sel berikut. Tidak ada path laptop yang ditanam di notebook.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import tempfile
import zipfile

BUNDLE_PATH = None  # Opsional: isi dengan path bundle yang kamu lampirkan.
candidate = Path(BUNDLE_PATH).expanduser() if BUNDLE_PATH else None
if candidate is None:
    kaggle = sorted(Path('/kaggle/input').glob('**/portfolio_lab_bundle.zip')) if Path('/kaggle/input').exists() else []
    local = Path.cwd() / 'portfolio_lab_bundle.zip'
    if kaggle:
        candidate = kaggle[0]
    elif local.is_file():
        candidate = local
    else:
        try:
            from google.colab import files
            uploaded = files.upload()
            names = [name for name in uploaded if name.endswith('.zip')]
            if len(names) == 1:
                candidate = Path(names[0])
        except ImportError:
            pass
if candidate is None or not candidate.is_file():
    raise FileNotFoundError('Lampirkan portfolio_lab_bundle.zip dari export lokal, lalu jalankan ulang sel ini.')

BUNDLE_ROOT = Path(tempfile.mkdtemp(prefix='factor-zoo-')) / 'bundle'
BUNDLE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(candidate) as archive:
    for member in archive.infolist():
        target = (BUNDLE_ROOT / member.filename).resolve()
        if BUNDLE_ROOT.resolve() not in target.parents and target != BUNDLE_ROOT.resolve():
            raise ValueError(f'Path tidak aman di dalam ZIP: {member.filename}')
    archive.extractall(BUNDLE_ROOT)
manifest = json.loads((BUNDLE_ROOT / 'manifest.json').read_text(encoding='utf-8'))
if manifest.get('provider') != 'Sectors.app' or manifest.get('external_api_calls') != 0:
    raise ValueError('Manifest bundle tidak cocok dengan kontrak sumber offline Sectors.app.')
for relative, expected_hash in manifest.get('source_files', {}).items():
    source = (BUNDLE_ROOT / relative).resolve()
    if BUNDLE_ROOT.resolve() not in source.parents or not source.is_file():
        raise ValueError(f'File sumber bundle hilang/tidak aman: {relative}')
    actual_hash = hashlib.sha256(source.read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f'Hash sumber tidak cocok: {relative}')
sys.path.insert(0, str(BUNDLE_ROOT / 'src'))
from idx_evidence_lab.portfolio_factors import build_factor_zoo_payload
print(f"Bundle siap: {manifest['ticker_count']} ticker, {manifest['file_count']} file, provider {manifest['provider']}")

## Kualitas dan cakupan

Status membedakan nilai terukur, faktor parsial, data yang tidak tersedia, dan nilai yang tidak layak dibandingkan. Contohnya, forward P/E ≤ 0 tidak dibalik menjadi earnings yield positif; nilai hilang tidak diisi nol. Komponen Quality distandardisasi terpisah untuk kelompok financial dan non-financial agar sektor bank tidak dibandingkan mentah dengan emiten lain.

In [ ]:
payload = build_factor_zoo_payload(BUNDLE_ROOT)
records = payload['records']
print('Snapshot as-of:', payload['as_of'])
print('Universe:', payload['coverage']['issuer_count'], 'emiten')
print('IHSG daily status:', payload['coverage']['index_daily_status'])
print('Data-source fingerprint:', payload['sources']['sha256'])

factor_names = ('value', 'quality', 'momentum', 'low_volatility')
for factor in factor_names:
    print(f"{factor}: {payload['coverage']['factors'][factor]}")

# Tabel komponen mempertahankan status/reason tiap ticker; jangan perlakukan kosong sebagai nol.
quality_rows = [{
    'ticker': row['ticker'], 'company_report': row['data_quality']['company_report'],
    'daily': row['data_quality']['daily'], 'aligned_IHSG_returns': row['data_quality']['aligned_ihsg_returns'],
    **{name: row['components'][name]['status'] for name in row['components']}
} for row in records]
try:
    import pandas as pd
    display(pd.DataFrame(quality_rows).sort_values('ticker').reset_index(drop=True))
except ImportError:
    for row in quality_rows:
        print(row)

## EDA

Periksa sebaran skor dan jumlah observasi valid sebelum membaca leaderboard. Ukuran sampel lintas emiten kecil dan universe snapshot memiliki survivorship/keanggotaan-terkini bias; distribusi ini bukan bukti premium faktor.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
for axis, factor in zip(axes.flat, factor_names):
    values = [row['scores'][factor]['score'] for row in records
              if row['scores'][factor]['score'] is not None]
    axis.hist(values, bins=12, color='#38bdf8', edgecolor='#0b1220')
    axis.axvline(0, color='#f59e0b', linestyle='--', linewidth=1)
    axis.set(title=f"{factor.replace('_', ' ').title()} · n={len(values)}", xlim=(-3, 3), xlabel='Z-score', ylabel='Jumlah emiten')
fig.suptitle('Sebaran skor faktor lintas emiten — deskriptif')
plt.show()

## Definisi faktor

- **Value:** earnings yield = 1 / forward P/E positif dan dividend yield TTM.
- **Quality:** ROE dan debt-to-equity; leverage dibalik arahnya. Distandardisasi dalam peer group financial/non-financial.
- **Momentum:** return harga 12–1 bulan (lewati kira-kira satu bulan terakhir) dan jarak harga terhadap MA200.
- **Low Volatility:** beta terhadap return IHSG dan volatilitas residual regresi harian; nilai beta/residual yang lebih rendah mendapat skor lebih tinggi.

Setiap komponen winsorized p5/p95, distandardisasi dengan rata-rata dan simpangan baku populasi lintas emiten, lalu dibatasi −3 sampai +3. Skor faktor adalah rerata komponen tersedia, bukan pengisian komponen hilang. Kurang dari lima peer valid atau variasi nol berarti abstain.

## Standardisasi — implementasi bersama

Tidak ada formula alternatif di notebook: data dan skor dihitung lewat fungsi yang sama dengan artefak MATLAB/prototipe agar hasilnya identik untuk input yang sama.

In [ ]:
score_rows = []
for row in records:
    entry = {'ticker': row['ticker'], 'company': row['company'], 'sector': row['sector'], 'sub_sector': row['sub_sector']}
    for factor in factor_names:
        item = row['scores'][factor]
        entry[f'{factor}_score'] = item['score']
        entry[f'{factor}_status'] = item['status']
    score_rows.append(entry)
if 'pd' in globals():
    factor_table = pd.DataFrame(score_rows).sort_values('ticker').reset_index(drop=True)
    display(factor_table)
else:
    factor_table = score_rows
    for row in score_rows:
        print(row)

## Leaderboard

Lima skor teratas per faktor ditampilkan hanya jika skornya tersedia penuh. Urutan ini menjelaskan posisi relatif pada snapshot aktif, bukan saham yang dipastikan unggul di masa depan.

In [ ]:
for factor in factor_names:
    print(f'\nTop {factor.replace("_", " ").title()}')
    rows = payload['leaderboards'][factor]
    if rows:
        for rank, item in enumerate(rows, start=1):
            print(f"{rank}. {item['ticker']}  score={item['score']:+.2f}  components={item['components']}  as-of={item['as_of']}")
    else:
        print('Belum cukup data valid untuk menyusun leaderboard.')

## Eksposur portofolio

Opsional: masukkan ticker dan bobot dari hasil Portfolio Lab. Eksposur dihitung sebagai rerata berbobot dari skor tersedia. Bobot dinormalisasi hanya di antara holdings yang skornya tersedia; jumlah cakupan ditampilkan agar hasil parsial tidak disamarkan.

In [ ]:
portfolio_weights = {}  # Contoh format input: {'BBCA': 0.40, 'BBRI': 0.35, 'TLKM': 0.25}
record_by_ticker = {row['ticker']: row for row in records}
for factor in factor_names:
    numerator = denominator = 0.0
    included = []
    for ticker, weight in portfolio_weights.items():
        score = record_by_ticker.get(ticker, {}).get('scores', {}).get(factor, {}).get('score')
        if score is not None and weight > 0:
            numerator += float(weight) * float(score)
            denominator += float(weight)
            included.append(ticker)
    if denominator:
        print(f"{factor}: {numerator / denominator:+.2f} | bobot tercakup {denominator:.1%} | {len(included)}/{len(portfolio_weights)} emiten")
    elif portfolio_weights:
        print(f'{factor}: unavailable — tidak ada holding dengan skor faktor yang valid')
    else:
        print(f'{factor}: masukkan portfolio_weights untuk melihat eksposur')

## Batas interpretasi

Factor Zoo ini potret lintas bagian (cross-section) dari data saat ini. Ia belum melakukan backtest, uji hipotesis, prediksi return, atau validasi out-of-sample. Snapshot keanggotaan sekarang tidak menghilangkan survivorship bias; laporan tanggal 25 Sep tidak otomatis berarti tersedia pada harga 24 Sep. Missing data dan pilihan field sumber bisa memengaruhi peringkat. Skor hanya alat eksplorasi dan **belum memengaruhi** optimasi bobot.

Jika sel belum dijalankan, seluruh hasil analisis tetap **PENDING / belum dijalankan**; jangan menyalin hasil numerik ke laporan sebelum notebook dijalankan dan input fingerprint diperiksa.

## Manifest

Tampilkan manifest file dan hash agar input run bisa diverifikasi serta direproduksi.

In [ ]:
from pprint import pprint
pprint({key: manifest.get(key) for key in ('provider', 'universe', 'ticker_count', 'file_count', 'hash_algorithm', 'source_files', 'limitations', 'external_api_calls')})
print('Analysis output status: READY only after this notebook is run against the attached local bundle.')